In [ ]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Read the API keys and configuration variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

In [ ]:
import datetime
import pandas as pd
from xgboost import XGBRegressor
import hopsworks
import json

today = datetime.datetime.now()
tomorrow = today + datetime.timedelta(days=1)

In [ ]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
city = location['city']

In [ ]:
mr = project.get_model_registry()

retrieved_model = mr.get_model(
    name="pollen_xgboost_model",
    version=1,
)

fv = retrieved_model.get_feature_view()
saved_model_dir = retrieved_model.download()

In [ ]:
retrieved_xgboost_model = XGBRegressor()
retrieved_xgboost_model.load_model(saved_model_dir + "/model.json")

In [ ]:
weather_fg = fs.get_feature_group(name='weather', version=1)
batch_data = weather_fg.filter(weather_fg.date >= today).read()

# Add temporal features
batch_data['day_of_year'] = batch_data['date'].dt.dayofyear
batch_data['month'] = batch_data['date'].dt.month
batch_data['is_high_season'] = batch_data['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# Add GDD
T_base = 5.0
batch_data['gdd_daily'] = batch_data['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
batch_data['gdd_cumsum'] = batch_data.groupby(batch_data['date'].dt.year)['gdd_daily'].cumsum()

# Add lagged features
batch_data['precip_lag_1'] = batch_data['precipitation_sum'].shift(1)
batch_data['temp_lag_1'] = batch_data['temperature_2m_mean'].shift(1)
batch_data['wind_lag_1'] = batch_data['wind_speed_10m_max'].shift(1)
batch_data = batch_data.dropna()
batch_data

In [ ]:
feature_cols = ['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
                'wind_direction_10m_dominant', 'day_of_year', 'month', 'is_high_season',
                'gdd_cumsum', 'precip_lag_1', 'temp_lag_1', 'wind_lag_1']

batch_data['predicted_pollen'] = retrieved_xgboost_model.predict(batch_data[feature_cols])
batch_data

In [ ]:
batch_data['city'] = city
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['date'])
batch_data

In [ ]:
monitor_fg = fs.get_or_create_feature_group(
    name='pollen_predictions',
    description='Pollen prediction monitoring',
    version=1,
    primary_key=['city','date','days_before_forecast_day'],
    event_time="date"
)

In [ ]:
monitor_fg.insert(batch_data, wait=True)

In [ ]:
monitoring_df = monitor_fg.filter(monitor_fg.days_before_forecast_day == 1).read()
monitoring_df

In [ ]:
pollen_fg = fs.get_feature_group(name='pollen', version=1)
pollen_df = pollen_fg.read()

outcome_df = pollen_df[['date', 'pollen_level']]
preds_df = monitoring_df[['date', 'predicted_pollen']]

hindcast_df = pd.merge(preds_df, outcome_df, on="date")
hindcast_df = hindcast_df.sort_values(by=['date'])
print(hindcast_df)